[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_34_Debate_Systems_with_Judge.ipynb)

# Lesson 34 — Debate Systems with a Judge

**Track 2 · Multi-Agent Coordination · Phase 4**

Last lesson (L33 — Blackboards) you saw what happens when you give N agents a *shared* workspace and a controller. The Editor was the **single decider** — only one knowledge source was allowed to flip `done=True`. Everyone else proposed, only one disposed.

Lesson 34 specializes that pattern into something the safety community has been chasing since 2018:

> **Debate** = a blackboard with two strictly-alternating *advocates* and a Judge that owns termination.

Two agents, told they are on opposing sides, take turns arguing. A third agent — the Judge — reads the transcript and picks a winner. Same architectural skeleton as L33 (workspace + actors + controller), but the activation rules are radically simpler: A goes, then B, then A, then B, until the turn budget runs out or the Judge calls it.

**What you'll build today:**
1. A `DebateBoard` data model (Pydantic) — the shared transcript
2. A `Debater(side, position)` class — Haiku, prompted to advocate, not to discover truth
3. A `Judge` class — Sonnet, the *only* actor with the authority to declare a winner
4. A `run_debate()` controller — enforces strict alternation and the turn budget
5. A **position-bias guard** — run the Judge twice with swapped order; if the verdict flips, it was bias, not substance
6. A failure-mode tour — Gish gallop, collusion, anchor-on-length, capability-mismatch
7. A wiring sketch that drops debaters into the L32 A2A network so each debater can live in its own process

Everything runs in one Colab kernel. No servers, no docker.

## 1 · Why debate? The 2018 AI-safety framing

Irving, Christiano & Amodei ("AI safety via debate", 2018) proposed a thought experiment:

> *Suppose two competing AIs argue both sides of a question, and a weaker human Judge picks a winner. Under adversarial cross-examination, the AIs are forced to either (a) tell the truth, because lies get exposed, or (b) at minimum surface the strongest counter-arguments. The Judge ends up better than either AI alone, even though the Judge is weaker than the AIs.*

The intuition is **adversarial information asymmetry**. A single Critic loop (L25) is one-sided — the Critic searches for flaws but doesn't *advocate*. A blackboard with a Critic KS (L33) is collaborative. **Debate is the only setup where each model has an explicit incentive to surface the strongest case for its assigned position.**

The Judge becomes the bottleneck. If the Judge is weaker than the debaters (the usual case — Sonnet judging Sonnets is fine; humans judging GPT-5s is the open research question), the debate format gives the weaker Judge leverage: instead of evaluating the truth itself, the Judge only has to evaluate *which advocate made the better case*, which is a much easier task.

**Where debate is the right tool:**

| Question type                                     | Right primitive   | Why                                                                |
| ------------------------------------------------- | ----------------- | ------------------------------------------------------------------ |
| Contested factual claim with two plausible sides  | **Debate**        | Adversarial advocacy surfaces the strongest objections             |
| One-sided revision ("make this draft better")   | Critic loop (L25) | No second position exists                                          |
| Multi-skill collaborative drafting                | Blackboard (L33)  | Multiple roles, no inherent adversarial axis                       |
| Pure information lookup                           | Single agent + tools | No reasoning under uncertainty                                    |

Pick debate when there are **two coherent positions** and your bottleneck is *evaluation*, not *generation*.

## 2 · Setup

We'll use Haiku for debaters (fast, cheap — they only need to advocate) and Sonnet for the Judge (the verdict requires sharper reasoning).

In [ ]:
!pip install anthropic pydantic httpx -q

In [ ]:
import os
from google.colab import userdata
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')

import anthropic
client = anthropic.Anthropic()

DEBATER_MODEL = 'claude-haiku-4-5'
JUDGE_MODEL   = 'claude-sonnet-4-5'   # judge needs a stronger reasoner

## 3 · The data model — `DebateBoard`

Just like L33's `Blackboard`, our debate has one shared, versioned data structure. The board is **append-only for turns** — once a debater speaks, that turn is frozen. The verdict is written exactly once, by the Judge.

Each `Turn` is structured: an explicit `claim`, optional `evidence` list, and an optional `refutes_turn` pointer back to the rival's earlier turn. Structured turns prevent the "wall of prose" antipattern where the Judge has to re-read everything to find the actual argument.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, Optional
from datetime import datetime, timezone

Side = Literal['A', 'B']
Phase = Literal['opening', 'rebuttal', 'closing']

class Turn(BaseModel):
    turn_id: int
    side: Side
    phase: Phase
    claim: str = Field(..., description="The single load-bearing claim of this turn")
    evidence: list[str] = Field(default_factory=list)
    refutes_turn: Optional[int] = Field(None, description="turn_id this rebuts, if any")
    raw_text: str = Field(..., description="The debater's full text (kept for transparency)")
    ts: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat())

class Verdict(BaseModel):
    winner: Literal['A', 'B', 'tie']
    confidence: float = Field(..., ge=0.0, le=1.0)
    rationale: str
    key_turns: list[int] = Field(default_factory=list, description="turn_ids the Judge says decided it")

class DebateBoard(BaseModel):
    topic: str
    side_A_position: str   # what A is arguing FOR
    side_B_position: str   # what B is arguing FOR
    turns: list[Turn] = Field(default_factory=list)
    verdict: Optional[Verdict] = None
    max_rounds: int = 3   # each side gets this many turns

    # ─── read helpers ───
    def turns_by(self, side: Side) -> list[Turn]:
        return [t for t in self.turns if t.side == side]

    def last_turn(self) -> Optional[Turn]:
        return self.turns[-1] if self.turns else None

    def next_side(self) -> Side:
        # strict alternation; A always opens
        if not self.turns:
            return 'A'
        return 'B' if self.turns[-1].side == 'A' else 'A'

    def next_phase(self, side: Side) -> Phase:
        n = len(self.turns_by(side))
        if n == 0:
            return 'opening'
        if n == self.max_rounds - 1:
            return 'closing'
        return 'rebuttal'

    def is_finished(self) -> bool:
        if self.verdict is not None:
            return True
        return (len(self.turns_by('A')) >= self.max_rounds
                and len(self.turns_by('B')) >= self.max_rounds)

    # ─── append helpers (the only mutators) ───
    def append_turn(self, turn: Turn) -> None:
        assert turn.side == self.next_side(), "strict alternation violated"
        self.turns.append(turn)

    def set_verdict(self, v: Verdict) -> None:
        assert self.verdict is None, "verdict already set — Judge is single-decider"
        self.verdict = v

**Why all the typing?** Same reason as L33: a typed transcript is *machine-checkable*. We can ask "did B ever refute turn 1?" or "which side cited the most evidence?" without re-running an LLM over the whole history. The Judge will appreciate it too — structured input has measurably less position bias than walls of prose.

Also notice the **invariant** in `append_turn`: strict alternation is encoded in the assert, not just in the prompt. The architecture refuses to enter an invalid state.

## 4 · Forcing structured output with `tool_choice`

Same trick as L33: we define a tool and force the model to call it. The tool input *is* our structured turn. This way we never have to write a brittle "please respond in JSON" prompt or parse from prose.

In [ ]:
import json

def call_with_tool(system: str, user: str, tool: dict, model: str, max_tokens: int = 1024) -> dict:
    """Call Anthropic with a single forced tool. Returns the tool input dict."""
    resp = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        system=system,
        tools=[tool],
        tool_choice={'type': 'tool', 'name': tool['name']},
        messages=[{'role': 'user', 'content': user}],
    )
    for block in resp.content:
        if block.type == 'tool_use':
            return block.input
    raise RuntimeError(f'no tool_use block returned: {resp}')

## 5 · The `Debater` — instructed to advocate, not to find truth

This is the most important design choice in a debate system. The Debater's job is **not** "answer the question as well as you can". It is "argue for *this assigned side*". The instruction to advocate is what makes the format adversarial.

**Three things the Debater system prompt must do:**
1. Pin the side. "You are arguing FOR side A: <position>." No wavering.
2. Refuse to concede. Even on rebuttal, the Debater patches its position rather than abandoning it.
3. Demand structure. The Debater must return `claim` + `evidence` + `refutes_turn` separately, so the Judge sees the bones.

**Two things it must NOT do:**
1. Generate harmful content because "the side requires it." The advocacy frame doesn't override safety; we keep the standard refusals on.
2. Strawman the opponent — we ask the Debater to *quote* the opponent's claim when refuting.

In [ ]:
submit_turn_tool = {
    'name': 'submit_turn',
    'description': 'Submit your structured debate turn',
    'input_schema': {
        'type': 'object',
        'properties': {
            'claim': {
                'type': 'string',
                'description': 'The single load-bearing claim you are making this turn (one sentence).',
            },
            'evidence': {
                'type': 'array',
                'items': {'type': 'string'},
                'description': '1–4 concrete evidence items (facts, examples, mechanisms). Do not invent citations.',
            },
            'refutes_turn': {
                'type': ['integer', 'null'],
                'description': 'turn_id of the opponent turn you are directly rebutting, or null on opening.',
            },
            'raw_text': {
                'type': 'string',
                'description': 'Your full prose argument for this turn (2–6 sentences).',
            },
        },
        'required': ['claim', 'evidence', 'refutes_turn', 'raw_text'],
    },
}

DEBATER_SYSTEM = """You are a debater in a structured debate.

You are arguing FOR side {side}: \"{position}\"

Rules of engagement:
1. You ADVOCATE for your assigned side. You do not pretend to be neutral.
2. You must NOT concede the debate. If your side has a weakness, find the strongest patch you can.
3. When you rebut, you must quote (or closely paraphrase) the opponent's specific claim and explain why it is wrong.
4. Evidence must be real. Do not invent statistics, papers, or quotes. If you don't know a concrete fact, reason mechanistically instead.
5. One load-bearing claim per turn. Don't Gish-gallop with 15 weak points.

Your assigned phase this turn is: {phase}
  - opening: state your main claim and 2–3 strongest reasons
  - rebuttal: directly attack the opponent's last turn, then advance your case
  - closing: summarize why you won; do NOT introduce new claims
"""

class Debater:
    def __init__(self, side: Side, position: str, model: str = DEBATER_MODEL):
        self.side = side
        self.position = position
        self.model = model

    def take_turn(self, board: DebateBoard) -> Turn:
        phase = board.next_phase(self.side)
        system = DEBATER_SYSTEM.format(side=self.side, position=self.position, phase=phase)
        user = self._build_user_message(board, phase)
        out = call_with_tool(system, user, submit_turn_tool, model=self.model)
        return Turn(
            turn_id=len(board.turns),
            side=self.side,
            phase=phase,
            claim=out['claim'],
            evidence=out.get('evidence', []),
            refutes_turn=out.get('refutes_turn'),
            raw_text=out['raw_text'],
        )

    def _build_user_message(self, board: DebateBoard, phase: Phase) -> str:
        lines = [f'TOPIC: {board.topic}',
                 f'YOUR POSITION (side {self.side}): {self.position}',
                 f'OPPONENT POSITION (side {"B" if self.side=="A" else "A"}): '
                 f'{board.side_B_position if self.side=="A" else board.side_A_position}',
                 '']
        if board.turns:
            lines.append('TRANSCRIPT SO FAR:')
            for t in board.turns:
                marker = '*' if t.side != self.side else ' '
                lines.append(f'  [{t.turn_id}] {marker} side {t.side} ({t.phase}): {t.claim}')
                if t.evidence:
                    lines.append(f'         evidence: {"; ".join(t.evidence)}')
        else:
            lines.append('(no turns yet — this is the opening)')
        lines.append('')
        lines.append(f'Now produce your {phase}. Call submit_turn.')
        return '\n'.join(lines)

## 6 · The `Judge` — single-decider, structured verdict

The Judge inherits the L33 **single-decider rule**: exactly one actor flips `done`. In debate terms, only the Judge writes the verdict.

The Judge's prompt must do three subtle things:
1. Read the transcript as turns, not as prose, so length doesn't anchor the verdict.
2. Score on **argument quality and refutation**, not on who *seems* more confident.
3. Allow `tie` — if neither side decisively wins, saying so is more honest than picking the longer one.

We also require the Judge to cite `key_turns` — which specific turn IDs decided the call. That's our audit trail. If next week someone challenges the verdict, we can replay exactly the turns the Judge leaned on.

In [ ]:
submit_verdict_tool = {
    'name': 'submit_verdict',
    'description': 'Submit your final verdict for the debate.',
    'input_schema': {
        'type': 'object',
        'properties': {
            'winner': {'type': 'string', 'enum': ['A', 'B', 'tie']},
            'confidence': {'type': 'number', 'minimum': 0.0, 'maximum': 1.0,
                           'description': 'How sure you are. ≤0.6 means it was close.'},
            'rationale': {'type': 'string',
                          'description': 'Why the winner won, 2–4 sentences, grounded in specific turns.'},
            'key_turns': {'type': 'array', 'items': {'type': 'integer'},
                          'description': 'turn_ids that decided the verdict.'},
        },
        'required': ['winner', 'confidence', 'rationale', 'key_turns'],
    },
}

JUDGE_SYSTEM = """You are an impartial judge of a structured debate.

You will see the topic, each side's position, and an ordered transcript of turns. Each turn has a structured claim, evidence list, optional refutation pointer, and raw text.

Rules of judgment:
1. Score on argument quality and refutation, NOT on confidence, length, or assertiveness.
2. Penalize Gish-gallop (many shallow claims) and reward focused, well-supported arguments.
3. Penalize fabricated evidence (numbers, papers, quotes that look invented).
4. Reward debaters who directly engaged the opponent's strongest point.
5. If neither side decisively won, say 'tie' — do not pick the longer transcript.
6. Cite specific turn_ids in key_turns. The verdict must be anchored to what was actually said.
"""

class Judge:
    def __init__(self, model: str = JUDGE_MODEL):
        self.model = model

    def render_transcript(self, board: DebateBoard, label_swap: bool = False) -> str:
        """Render the transcript. If label_swap, present B as 'Side A' and vice versa.
        Used by the position-bias guard."""
        def relabel(s: Side) -> str:
            if not label_swap:
                return s
            return 'B' if s == 'A' else 'A'

        a_pos = board.side_A_position
        b_pos = board.side_B_position
        if label_swap:
            a_pos, b_pos = b_pos, a_pos

        lines = [f'TOPIC: {board.topic}',
                 f'Side A position: {a_pos}',
                 f'Side B position: {b_pos}',
                 '',
                 'TRANSCRIPT:']
        for t in board.turns:
            label = relabel(t.side)
            lines.append(f'[turn {t.turn_id}] side {label} ({t.phase})')
            lines.append(f'    claim:   {t.claim}')
            if t.evidence:
                for e in t.evidence:
                    lines.append(f'    -        {e}')
            if t.refutes_turn is not None:
                lines.append(f'    rebuts:  turn {t.refutes_turn}')
            lines.append('')
        return '\n'.join(lines)

    def deliberate(self, board: DebateBoard, label_swap: bool = False) -> Verdict:
        user = self.render_transcript(board, label_swap=label_swap)
        out = call_with_tool(JUDGE_SYSTEM, user, submit_verdict_tool, model=self.model, max_tokens=512)
        v = Verdict(**out)
        if label_swap and v.winner in ('A', 'B'):
            v = v.model_copy(update={'winner': 'B' if v.winner == 'A' else 'A'})
        return v

## 7 · The controller — strict alternation, hard turn budget

Compare this to L33's controller. L33 had four KSs and a *priority list*; the scheduler had to decide who acts next. Here the answer is mechanical: "whose turn is it next?" The controller is 15 lines.

That simplicity is the whole point. **Debate is the cheap end of multi-agent coordination.** No deadlock detection, no anti-monopoly guards, no last-actor checks — strict alternation makes all of that trivially safe.

In [ ]:
def run_debate(board: DebateBoard, debater_A: Debater, debater_B: Debater,
               judge: Judge, *, position_bias_guard: bool = True, verbose: bool = True) -> DebateBoard:
    debaters = {'A': debater_A, 'B': debater_B}
    while not board.is_finished():
        side = board.next_side()
        if verbose:
            print(f'─── turn {len(board.turns)}: side {side} ({board.next_phase(side)}) ───')
        turn = debaters[side].take_turn(board)
        board.append_turn(turn)
        if verbose:
            print(f'  claim:    {turn.claim}')
            if turn.evidence:
                print(f'  evidence: {len(turn.evidence)} items')
            if turn.refutes_turn is not None:
                print(f'  rebuts:   turn {turn.refutes_turn}')

    # ── judge phase ──
    if verbose:
        print('\n─── judge deliberating ───')
    v_primary = judge.deliberate(board, label_swap=False)

    if position_bias_guard:
        if verbose:
            print('─── judge deliberating again with swapped labels (bias guard) ───')
        v_swapped = judge.deliberate(board, label_swap=True)
        if v_primary.winner != v_swapped.winner:
            # the verdict flipped — bias detected, downgrade to tie
            v_final = Verdict(
                winner='tie',
                confidence=min(v_primary.confidence, v_swapped.confidence) * 0.5,
                rationale=(f'POSITION BIAS DETECTED. Primary verdict picked '
                           f'{v_primary.winner}; swapped-label verdict picked '
                           f'{v_swapped.winner}. Forcing tie.'),
                key_turns=sorted(set(v_primary.key_turns + v_swapped.key_turns)),
            )
        else:
            # agreed — keep primary verdict but average the confidence
            v_final = v_primary.model_copy(update={
                'confidence': (v_primary.confidence + v_swapped.confidence) / 2,
            })
    else:
        v_final = v_primary

    board.set_verdict(v_final)
    return board

## 8 · Live run — a contested factual claim

We pick a topic with two coherent sides and unambiguous evaluation criteria. Avoid pure value debates ("is X morally good") — they reward rhetoric over reasoning, which is exactly what we don't want to demo on day one.

**Topic:** *Should a small SaaS startup default to a relational database over a document store for its first product?*

Both sides are defensible; the answer depends on assumptions about schema stability, transactionality, and team familiarity. Perfect debate territory.

In [ ]:
board = DebateBoard(
    topic='Should a small SaaS startup default to a relational database over a document store for its first product?',
    side_A_position='YES — Postgres-by-default beats Mongo-by-default for nearly every early-stage SaaS.',
    side_B_position='NO — A document store like MongoDB is the better default for early-stage SaaS.',
    max_rounds=3,
)

debater_A = Debater('A', board.side_A_position)
debater_B = Debater('B', board.side_B_position)
judge = Judge()

board = run_debate(board, debater_A, debater_B, judge, position_bias_guard=True, verbose=True)

### Inspect the verdict

In [ ]:
print(f'WINNER:     {board.verdict.winner}')
print(f'CONFIDENCE: {board.verdict.confidence:.2f}')
print(f'KEY TURNS:  {board.verdict.key_turns}')
print()
print('RATIONALE:')
print(board.verdict.rationale)

### Read the actual transcript

In [ ]:
for t in board.turns:
    print(f'── turn {t.turn_id} · side {t.side} · {t.phase} ──')
    print(f'CLAIM: {t.claim}')
    if t.refutes_turn is not None:
        print(f'(rebuts turn {t.refutes_turn})')
    if t.evidence:
        for e in t.evidence:
            print(f'  • {e}')
    print()
    print(t.raw_text)
    print()

## 9 · The position-bias guard, in detail

LLM judges have a well-documented **position bias**: when shown two options labeled "A" and "B", many models prefer "A" *more often than chance*, even when A and B are identical. The bias varies by model, prompt, and topic — and you cannot prompt it away reliably.

The fix is the same one we used in L25's `pairwise_judge`: **run the Judge twice with swapped labels** and require agreement. If the two runs disagree, you've detected bias and you should downgrade to `tie`.

Our `run_debate` does this automatically. Let's see what it looks like when you force a verdict-only call:

In [ ]:
# Manually re-run both Judge passes on the same board to see the raw verdicts
v_primary = judge.deliberate(board, label_swap=False)
v_swapped = judge.deliberate(board, label_swap=True)

print('Primary verdict (labels unchanged): winner', v_primary.winner,
      'conf', round(v_primary.confidence, 2))
print('Swapped  verdict (labels flipped):  winner', v_swapped.winner,
      'conf', round(v_swapped.confidence, 2))
print()
if v_primary.winner == v_swapped.winner:
    print('✅ Verdicts agree — substantive, not bias-driven.')
else:
    print('⚠️  Verdicts disagree — position bias detected.')

**💡 EXPERIMENT:** Run the swap multiple times. Verdicts can drift with sampling temperature even when both agree at this temperature. In production you'd run *N* swapped pairs and require majority agreement, not just a single round-trip.

## 10 · Failure modes — the dark side of debate

Debate looks elegant in the abstract. In practice these failure modes show up almost immediately:

| Failure                             | What it looks like                                                                 | Mitigation                                                                                              |
| ----------------------------------- | ---------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------- |
| **Gish gallop**                     | One debater spams 12 weak claims; opponent and Judge run out of budget to rebut    | Cap evidence items per turn; explicit single-claim rule in the Debater prompt                          |
| **Collusion**                       | Both debaters drift toward the same answer (same training data, same priors)       | Different model families (e.g. Sonnet vs Haiku, or Claude vs Llama); explicit "do not concede" rule    |
| **Length anchoring**                | Judge picks the longer transcript even when content is weaker                      | Structured turns; tell Judge explicitly that length is not evidence                                     |
| **Capability mismatch**             | Sonnet routinely beats Haiku regardless of side. The Judge sees prose quality, not argument quality. | Match debater models; or normalize for capability by also evaluating the Debater's *task*, not just its output |
| **Position bias in the Judge**      | A wins more than 50% on identical content                                          | The label-swap guard we just built                                                                      |
| **Topic that has no two sides**     | One side is just wrong ("the earth is flat"). Debate gives a flat-earther airtime. | Pre-screen topics; for genuinely-contested claims only                                                  |
| **Premature closing**               | The closing happens before key arguments are explored                              | Tune `max_rounds`; or add a "requires_more_rounds" signal from Judge                                  |

**The capability-mismatch one is the most underappreciated.** A 2024 study had a strong model debate a weak model and let a strong-model Judge pick the winner — the verdicts tracked model capability much more than argument quality. Debate only gives the Judge leverage when both debaters are *capable enough to advocate*. If one debater is fundamentally outmatched, you've just built an expensive way to rerun the strong model.

## 11 · Debate vs Critic Loop vs Blackboard — when to pick which

You now have three multi-agent reasoning patterns. The decision is mostly about **what kind of bottleneck you're trying to break**:

| Pattern                  | Number of voices | Mode of disagreement      | Best for                                              | Cost                                  |
| ------------------------ | ---------------- | -------------------------- | ----------------------------------------------------- | ------------------------------------- |
| **Critic loop (L25)**    | 1 author + 1 critic | one-sided revision        | Polishing a single document; catching factual errors  | ~2× single-agent                      |
| **Blackboard (L33)**     | N specialists + Editor | collaborative drafting  | Multi-skill workflows (search + synthesize + edit)    | N × single-agent (worth it for breadth) |
| **Debate (L34)**         | 2 advocates + Judge | adversarial advocacy      | Two-sided factual/strategic questions; eval-bound problems | (2 × rounds + 2 × Judge passes) × single-agent |

**Quick decision tree:**
1. Is the bottleneck *generation* (you have to produce many things)? → Blackboard.
2. Is the bottleneck *quality control* on a single output? → Critic loop.
3. Is the bottleneck *evaluation* under genuine uncertainty between two coherent positions? → Debate.

Notice debate is the most expensive of the three. Use it when the evaluation problem is the bottleneck — otherwise the cheaper patterns dominate.

## 12 · Mini-capstone — wire debaters into the L32 A2A network

From L33 you already know the punchline: **a `Debater` is just a `KnowledgeSource` is just a function**. The controller doesn't care if the Debater runs in this Python process, in another process behind FastAPI, or on a different machine entirely. As long as it returns a `Turn`, the controller proceeds.

Below is a sketch (don't run it — it assumes two A2A servers from L32 are already running on ports 8001 and 8002, which we don't spin up in this notebook). It's the load-bearing 25 lines you'd swap into a real deployment.

In [ ]:
# ── sketch only — uncomment and run if you have L32's A2A servers up ──
# import httpx, uuid
# 
# class A2ADebater(Debater):
#     def __init__(self, side, position, agent_url):
#         super().__init__(side, position)
#         self.agent_url = agent_url  # e.g. http://localhost:8001
#
#     def take_turn(self, board: DebateBoard) -> Turn:
#         phase = board.next_phase(self.side)
#         payload = {
#             'id': str(uuid.uuid4()),               # idempotency
#             'message': {
#                 'role': 'user',
#                 'parts': [{'kind': 'data', 'data': {
#                     'topic': board.topic,
#                     'side': self.side,
#                     'position': self.position,
#                     'phase': phase,
#                     'transcript': [t.model_dump() for t in board.turns],
#                 }}],
#             }
#         }
#         r = httpx.post(f'{self.agent_url}/tasks/send', json=payload, timeout=30)
#         task_id = r.json()['id']
#         # poll until terminal — would normally factor into A2AClient.wait_until_done
#         while True:
#             status = httpx.get(f'{self.agent_url}/tasks/{task_id}').json()
#             if status['status']['state'] in ('completed', 'failed', 'canceled'):
#                 break
#         artifact = status['artifacts'][-1]
#         data = next(p['data'] for p in artifact['parts'] if p['kind'] == 'data')
#         return Turn(turn_id=len(board.turns), side=self.side, phase=phase, **data)

# Then the rest of the system is unchanged:
# debater_A = A2ADebater('A', board.side_A_position, agent_url='http://localhost:8001')
# debater_B = A2ADebater('B', board.side_B_position, agent_url='http://localhost:8002')
# judge     = Judge()  # still in-process; the Judge holds the verdict invariant
# board     = run_debate(board, debater_A, debater_B, judge)
print('A2A wiring sketch — see comments above. Same Controller, debaters now remote.')

**The architecturally interesting fact:** the Controller, the position-bias guard, and the verdict invariant are all *unchanged*. The same code orchestrates local Python objects, in-process functions, *and* remote A2A debaters — exactly the property that made L33's controller composable across deployment topologies.

## 13 · Replay & audit

Because every turn is structured and the verdict cites `key_turns`, we can replay the debate's decisive path in three lines:

In [ ]:
print('═══ DECISIVE PATH (cited by Judge) ═══')
for tid in board.verdict.key_turns:
    t = board.turns[tid]
    print(f'[{tid}] side {t.side} ({t.phase}): {t.claim}')
print()
print(f'verdict: {board.verdict.winner}  ·  confidence: {board.verdict.confidence:.2f}')

**Why this matters for production:** when an unhappy stakeholder asks "why did the system pick this answer?", you point to the verdict's `key_turns` and the `rationale`. No reconstructing a wall of prose. The reason is in the data model.

This is the same principle as L33's per-write `Action` audit log, specialized to debate. Both lessons hammered the same point: **make decisions inspectable by recording them as data, not as prose.**

## 14 · Homework

1. **Add a Verifier turn.** After the Judge issues its primary verdict but before the swapped-label check, insert a `Verifier` agent (Haiku) that *only* checks whether the Judge's `key_turns` actually contain what the Judge says they contain. If the Verifier says no, force `tie`. This catches Judge hallucination, which is a real failure mode.

2. **Score-based termination.** Replace the fixed `max_rounds` budget with a Judge-driven "continue or stop?" signal after each pair of turns. The Judge returns `continue=True` if it still can't pick a winner. Be careful: if you let the Judge run forever, debaters will lap themselves.

3. **Pairwise position bias measurement.** Build a small harness: pick 10 controversial topics, run 5 debates each, then run the Judge with and without label-swap. Compute the fraction of label-swap rounds that flip the verdict. That's your *measured* bias rate — useful as an eval gate.

4. **Asymmetric models.** Run a Sonnet debater against a Haiku debater on the same topic. Does the Sonnet side win every time? Use this to demonstrate the capability-mismatch failure mode to yourself.

5. **A2A-ize one debater.** Take L32's A2A server skeleton and host the `A2ADebater` for real, on a second port. Confirm that `run_debate` doesn't change. The Controller-doesn't-care-where-the-actor-lives property is the most important architectural takeaway of Track 2 so far.

## 15 · Coming up — Lesson 35

**Track 2 · Parallel Fan-out & Map-Reduce across A2A Peers**

So far in Track 2 we've built sequential coordination: A2A (1→1), Blackboard (round-robin), Debate (strictly alternating). Lesson 35 introduces the **parallel** dimension: one orchestrator fans a task out to N workers concurrently and reduces their outputs.

We'll cover:
- `asyncio.gather` for in-process fan-out
- Distributed fan-out across A2A peers (same `httpx.post`, but N at once)
- The map-reduce shape for embarrassingly-parallel agent work (per-document summarization, per-claim verification)
- Partial-failure semantics — what if 7 of 10 workers return successfully?
- Cost amplification (N× the cost of single-agent) and when it's worth it

After L35 you'll have all four primitive coordination shapes — sequential pairs (A2A), shared workspace (Blackboard), adversarial advocacy (Debate), parallel fan-out (next). L36 is the Track 2 capstone where we wire all four together into a single research swarm.

See you tomorrow.